In [10]:
# %%
import os
import re
from pathlib import Path
from collections import defaultdict

import yaml
from dotenv import load_dotenv
import psycopg2
from psycopg2 import sql

In [11]:
# ----------------------------
# paths
# ----------------------------
project_root = Path.cwd().parents[0]
config_path = project_root / "configs" / "paths.yaml"

with open(config_path) as f:
    paths_cfg = yaml.safe_load(f)

data_external = project_root / paths_cfg["data"]["external"]

DEM_DIR = data_external / "GEE_exportsDEM"

In [12]:
# ----------------------------
# db connect
# ----------------------------
load_dotenv()

conn = psycopg2.connect(
    dbname=os.getenv("PGDATABASE"),
    user=os.getenv("PGUSER"),
    password=os.getenv("PGPASSWORD"),
    host=os.getenv("PGHOST"),
    port=os.getenv("PGPORT"),
)
conn.autocommit = False

In [13]:
def run_sql(query, params=None):
    with conn.cursor() as cur:
        cur.execute(query, params)

def ensure_schema(schema_name: str):
    run_sql(sql.SQL("CREATE SCHEMA IF NOT EXISTS {}").format(sql.Identifier(schema_name)))

def drop_table(schema_name: str, table_name: str):
    run_sql(sql.SQL("DROP TABLE IF EXISTS {}.{} CASCADE").format(
        sql.Identifier(schema_name), sql.Identifier(table_name)
    ))

def drop_view(schema_name: str, view_name: str):
    run_sql(sql.SQL("DROP VIEW IF EXISTS {}.{} CASCADE").format(
        sql.Identifier(schema_name), sql.Identifier(view_name)
    ))

def get_relation_kind(schema_name: str, relation_name: str):
    q = """
    SELECT c.relkind
    FROM pg_class c
    JOIN pg_namespace n ON n.oid = c.relnamespace
    WHERE n.nspname = %s AND c.relname = %s
    """
    with conn.cursor() as cur:
        cur.execute(q, (schema_name, relation_name))
        row = cur.fetchone()
        return row[0] if row else None

def drop_relation(schema_name: str, relation_name: str):
    """Drop an existing table, view, or materialized view with the given name."""
    relkind = get_relation_kind(schema_name, relation_name)
    if relkind in ('v',):
        drop_view(schema_name, relation_name)
    elif relkind in ('r', 'p'):
        drop_table(schema_name, relation_name)
    elif relkind == 'm':
        run_sql(sql.SQL("DROP MATERIALIZED VIEW IF EXISTS {}.{} CASCADE").format(
            sql.Identifier(schema_name), sql.Identifier(relation_name)
        ))

def table_exists(schema: str, table: str) -> bool:
    q = """
    SELECT EXISTS (
        SELECT 1
        FROM information_schema.tables
        WHERE table_schema = %s AND table_name = %s
    )
    """
    with conn.cursor() as cur:
        cur.execute(q, (schema, table))
        return cur.fetchone()[0]

def get_columns(schema: str, table: str) -> list[str]:
    q = """
        SELECT column_name
        FROM information_schema.columns
        WHERE table_schema = %s AND table_name = %s
        ORDER BY ordinal_position
    """
    with conn.cursor() as cur:
        cur.execute(q, (schema, table))
        return [r[0] for r in cur.fetchall()]
def create_joined_output_two_sources(
    view_name: str,
    dem_schema: str, dem_table: str,
    ag_schema: str,  ag_table: str,
    output_schema: str, output_table: str,
    dem_prefix: str = "dem_",
    ag_prefix: str = "ag_",
):
    """
    Create ONE output table by joining geometry table to DEM accum + AG staging/accum.
    """
    ensure_schema(output_schema)
    drop_table(output_schema, output_table)

    dem_cols = [c for c in get_columns(dem_schema, dem_table) if c != ID_COL]
    ag_cols  = [c for c in get_columns(ag_schema,  ag_table)  if c != ID_COL]

    dem_select = sql.SQL(", ").join(
        sql.SQL("d.{} AS {}").format(
            sql.Identifier(c),
            sql.Identifier(f"{dem_prefix}{c}")
        ) for c in dem_cols
    )

    ag_select = sql.SQL(", ").join(
        sql.SQL("a.{} AS {}").format(
            sql.Identifier(c),
            sql.Identifier(f"{ag_prefix}{c}")
        ) for c in ag_cols
    )

    # Handle empty selects cleanly
    extra_selects = [s for s in [dem_select, ag_select] if str(s) != ""]
    extras_sql = sql.SQL(", ") + sql.SQL(", ").join(extra_selects) if extra_selects else sql.SQL("")

    join_sql = sql.SQL("""
        CREATE TABLE {}.{} AS
        SELECT
            v.*{}
        FROM {}.{} AS v
        LEFT JOIN {}.{} AS d
          ON v.{} = NULLIF(d.{}, '')::integer
        LEFT JOIN {}.{} AS a
          ON v.{} = NULLIF(a.{}, '')::integer;
    """).format(
        sql.Identifier(output_schema),
        sql.Identifier(output_table),
        extras_sql,
        sql.Identifier(SCHEMA),
        sql.Identifier(view_name),
        sql.Identifier(dem_schema),
        sql.Identifier(dem_table),
        sql.Identifier(ID_COL),
        sql.Identifier(ID_COL),
        sql.Identifier(ag_schema),
        sql.Identifier(ag_table),
        sql.Identifier(ID_COL),
        sql.Identifier(ID_COL),
    )

    run_sql(join_sql)

    run_sql(sql.SQL("CREATE INDEX ON {}.{} ({})").format(
        sql.Identifier(output_schema),
        sql.Identifier(output_table),
        sql.Identifier(ID_COL),
    ))


In [14]:
# ----------------------------
# config
# ----------------------------
SCHEMA = "public"          # geometry tables schema
STAGING_SCHEMA = "scratch" # staging schema for csv loads
OUTPUT_SCHEMA = "public"   # output joined tables schema
OUTPUT_PREFIX = "joined_"

ID_COL = "OID"             # join key in both tables + csvs

# Metadata columns present in new chunked GEE exports that should
# be dropped before accumulation (only OID + data cols are kept).
META_COLS_DROP = frozenset({
    "system:index", "ADM0_EN", "ADM1_EN", "ADM2_EN",
    "bucket", "chunk_id", "lat", "lon", "rand", ".geo",
    "name", "settlement",
})

# Map (resolution, type) -> target geometry table  [15k pipeline]
TARGET_TABLES = {
    (15, "grids"):   "grid_15k_v3",
    (15, "hexbins"): "hex_15k"
}

# AG files live directly in data_external, not chunked
AG_FILES = [
    "AG_15_grids.csv",
    "AG_15_hexbins.csv"
]

# DEM chunks regex: DEM_##_type#.csv
DEM_RE = re.compile(r"^DEM_(\d+)_((?:grids)|(?:hexbins))(\d+)\.csv$", re.IGNORECASE)

In [15]:
# ----------------------------
# csv loading helpers
# ----------------------------
def copy_csv_to_table(csv_path: Path, schema_name: str, table_name: str):
    """
    Create table with TEXT columns from header and COPY data in.
    """
    header = csv_path.read_text(encoding="utf-8", errors="replace").splitlines()[0]
    cols = [c.strip() for c in header.split(",")]

    if ID_COL not in cols:
        raise ValueError(f"{csv_path.name} missing required id column '{ID_COL}'")

    ensure_schema(schema_name)
    drop_table(schema_name, table_name)

    col_defs = sql.SQL(", ").join([
        sql.SQL("{} TEXT").format(sql.Identifier(c)) for c in cols
    ])

    run_sql(sql.SQL("CREATE UNLOGGED TABLE {}.{} ({});").format(
        sql.Identifier(schema_name), sql.Identifier(table_name), col_defs
    ))

    with conn.cursor() as cur, open(csv_path, "r", encoding="utf-8") as f:
        copy_cmd = sql.SQL(
            "COPY {}.{} ({}) FROM STDIN WITH (FORMAT csv, HEADER true)"
        ).format(
            sql.Identifier(schema_name),
            sql.Identifier(table_name),
            sql.SQL(", ").join(map(sql.Identifier, cols))
        )
        cur.copy_expert(copy_cmd.as_string(cur), f)


def get_data_cols(csv_path: Path, drop_cols: frozenset = META_COLS_DROP) -> list[str]:
    """
    Read the CSV header and return columns after removing metadata columns.
    OID is always kept.
    """
    header = csv_path.read_text(encoding="utf-8", errors="replace").splitlines()[0]
    all_cols = [c.strip() for c in header.split(",")]
    kept = [c for c in all_cols if c not in drop_cols]
    if ID_COL not in kept:
        raise ValueError(f"{csv_path.name}: '{ID_COL}' not present after filtering metadata")
    return kept


def copy_csv_to_table_filtered(
    csv_path: Path,
    schema_name: str,
    table_name: str,
    drop_cols: frozenset = META_COLS_DROP,
) -> list[str]:
    """
    Load a CSV into a staging table, keeping only OID + data columns
    (i.e. stripping all columns in drop_cols).  Returns the list of kept columns.
    """
    keep_cols = get_data_cols(csv_path, drop_cols)

    # Stage the full CSV first, then INSERT SELECT only the kept columns.
    tmp_table = f"_tmp_{table_name}"
    copy_csv_to_table(csv_path, schema_name, tmp_table)

    ensure_schema(schema_name)
    drop_table(schema_name, table_name)

    col_defs = sql.SQL(", ").join(
        sql.SQL("{} TEXT").format(sql.Identifier(c)) for c in keep_cols
    )
    run_sql(sql.SQL("CREATE UNLOGGED TABLE {}.{} ({});").format(
        sql.Identifier(schema_name), sql.Identifier(table_name), col_defs
    ))

    col_list = sql.SQL(", ").join(map(sql.Identifier, keep_cols))
    run_sql(sql.SQL("INSERT INTO {}.{} ({}) SELECT {} FROM {}.{};").format(
        sql.Identifier(schema_name), sql.Identifier(table_name), col_list,
        col_list,
        sql.Identifier(schema_name), sql.Identifier(tmp_table),
    ))

    drop_table(schema_name, tmp_table)
    return keep_cols


def sorted_chunks(directory: Path, pattern: "re.Pattern") -> list[Path]:
    """
    Return all CSVs in directory whose names match pattern, sorted by
    the integer captured in group 1 of the pattern.
    """
    result = []
    for p in directory.glob("*.csv"):
        m = pattern.match(p.name)
        if m:
            result.append((int(m.group(1)), p))
    return [p for _, p in sorted(result)]


def ensure_accum_table(acc_schema: str, acc_table: str, cols: list[str]):
    """
    Ensure an accumulation table exists with TEXT columns, and a unique index on ID_COL.
    """
    ensure_schema(acc_schema)
    if not table_exists(acc_schema, acc_table):
        col_defs = sql.SQL(", ").join([
            sql.SQL("{} TEXT").format(sql.Identifier(c)) for c in cols
        ])
        run_sql(sql.SQL("CREATE UNLOGGED TABLE {}.{} ({});").format(
            sql.Identifier(acc_schema), sql.Identifier(acc_table), col_defs
        ))
        run_sql(sql.SQL("CREATE UNIQUE INDEX ON {}.{} ({})").format(
            sql.Identifier(acc_schema), sql.Identifier(acc_table), sql.Identifier(ID_COL)
        ))

def upsert_chunk_into_accum(cols: list[str],
                            stg_schema: str, stg_table: str,
                            acc_schema: str, acc_table: str):
    non_id_cols = [c for c in cols if c != ID_COL]

    insert_cols = sql.SQL(", ").join(map(sql.Identifier, cols))
    outer_select_cols = sql.SQL(", ").join([sql.SQL("d.{}").format(sql.Identifier(c)) for c in cols])

    set_clause = sql.SQL(", ").join([
        sql.SQL("{} = EXCLUDED.{}").format(sql.Identifier(c), sql.Identifier(c))
        for c in non_id_cols
    ])

    inner_select_cols = sql.SQL(", ").join([sql.Identifier(c) for c in cols])

    q = sql.SQL("""
        INSERT INTO {}.{} ({})
        SELECT {}
        FROM (
            SELECT DISTINCT ON ({})
                {}
            FROM {}.{} AS s
            WHERE NULLIF(s.{}, '') IS NOT NULL
            ORDER BY {}
        ) AS d
        ON CONFLICT ({}) DO UPDATE SET {};
    """).format(
        sql.Identifier(acc_schema),
        sql.Identifier(acc_table),
        insert_cols,
        outer_select_cols,
        sql.Identifier(ID_COL),
        inner_select_cols,
        sql.Identifier(stg_schema),
        sql.Identifier(stg_table),
        sql.Identifier(ID_COL),
        sql.Identifier(ID_COL),
        sql.Identifier(ID_COL),
        set_clause
    )
    run_sql(q)

    
def load_ag_to_accum(csv_path: Path, acc_schema: str, acc_table: str):
    """
    Load AG CSV into a deduped accumulation table with 1 row per OID.
    Keeps the first row per OID (deterministic: ORDER BY OID).
    """
    # 1) load raw into a staging table
    stg_table = f"stg_{csv_path.stem.lower()}"
    copy_csv_to_table(csv_path, acc_schema, stg_table)

    # 2) build deduped table (drop/recreate)
    header = csv_path.read_text(encoding="utf-8", errors="replace").splitlines()[0]
    cols = [c.strip() for c in header.split(",")]
    if ID_COL not in cols:
        raise ValueError(f"{csv_path.name} missing required id column '{ID_COL}'")

    ensure_schema(acc_schema)
    drop_table(acc_schema, acc_table)

    # create deduped table with same TEXT columns
    col_defs = sql.SQL(", ").join([sql.SQL("{} TEXT").format(sql.Identifier(c)) for c in cols])
    run_sql(sql.SQL("CREATE UNLOGGED TABLE {}.{} ({});").format(
        sql.Identifier(acc_schema), sql.Identifier(acc_table), col_defs
    ))

    insert_cols = sql.SQL(", ").join(map(sql.Identifier, cols))
    select_cols = sql.SQL(", ").join([sql.SQL("d.{}").format(sql.Identifier(c)) for c in cols])
    inner_cols  = sql.SQL(", ").join([sql.Identifier(c) for c in cols])

    dedupe_insert = sql.SQL("""
        INSERT INTO {}.{} ({})
        SELECT {}
        FROM (
            SELECT DISTINCT ON ({})
                {}
            FROM {}.{} AS s
            WHERE NULLIF(s.{}, '') IS NOT NULL
            ORDER BY {}
        ) AS d;
    """).format(
        sql.Identifier(acc_schema),
        sql.Identifier(acc_table),
        insert_cols,
        select_cols,
        sql.Identifier(ID_COL),
        inner_cols,
        sql.Identifier(acc_schema),
        sql.Identifier(stg_table),
        sql.Identifier(ID_COL),
        sql.Identifier(ID_COL),
    )

    run_sql(dedupe_insert)

    run_sql(sql.SQL("CREATE UNIQUE INDEX ON {}.{} ({})").format(
        sql.Identifier(acc_schema),
        sql.Identifier(acc_table),
        sql.Identifier(ID_COL),
    ))

    drop_table(acc_schema, stg_table)


def load_flat_to_accum(
    csv_path: Path,
    acc_schema: str,
    acc_table: str,
    drop_cols: frozenset = META_COLS_DROP,
):
    """
    Like load_ag_to_accum but strips metadata columns before accumulating.
    Used for the AG_DEM_250 flat files.
    """
    stg_table = f"stg_{csv_path.stem.lower()}"
    keep_cols = copy_csv_to_table_filtered(csv_path, acc_schema, stg_table, drop_cols)

    if ID_COL not in keep_cols:
        raise ValueError(f"{csv_path.name}: '{ID_COL}' missing after column filter")

    ensure_schema(acc_schema)
    drop_table(acc_schema, acc_table)

    col_defs = sql.SQL(", ").join(
        sql.SQL("{} TEXT").format(sql.Identifier(c)) for c in keep_cols
    )
    run_sql(sql.SQL("CREATE UNLOGGED TABLE {}.{} ({});").format(
        sql.Identifier(acc_schema), sql.Identifier(acc_table), col_defs
    ))

    insert_cols = sql.SQL(", ").join(map(sql.Identifier, keep_cols))
    select_cols = sql.SQL(", ").join(
        sql.SQL("d.{}").format(sql.Identifier(c)) for c in keep_cols
    )
    inner_cols = sql.SQL(", ").join(map(sql.Identifier, keep_cols))

    run_sql(sql.SQL("""
        INSERT INTO {}.{} ({})
        SELECT {}
        FROM (
            SELECT DISTINCT ON ({})
                {}
            FROM {}.{} AS s
            WHERE NULLIF(s.{}, '') IS NOT NULL
            ORDER BY {}
        ) AS d;
    """).format(
        sql.Identifier(acc_schema), sql.Identifier(acc_table), insert_cols,
        select_cols,
        sql.Identifier(ID_COL),
        inner_cols,
        sql.Identifier(acc_schema), sql.Identifier(stg_table),
        sql.Identifier(ID_COL),
        sql.Identifier(ID_COL),
    ))

    run_sql(sql.SQL("CREATE UNIQUE INDEX ON {}.{} ({})").format(
        sql.Identifier(acc_schema), sql.Identifier(acc_table), sql.Identifier(ID_COL)
    ))

    drop_table(acc_schema, stg_table)

In [16]:
# ----------------------------
# 250k pipeline config
# ----------------------------

# Each entry: (source_name, geom_type, directory, file_regex, col_prefix)
# file_regex must capture the chunk number in group 1.
# col_prefix is prepended to every data column in the final joined table.
CHUNK_SOURCE_CONFIGS_250 = [
    ("spei",   "grids",
     data_external / "SPEI" / "250",
     re.compile(r"^SPEI_250_grids(\d+)\.csv$",             re.IGNORECASE),
     "spei_"),
    ("spei",   "hexbins",
     data_external / "SPEI" / "250",
     re.compile(r"^SPEI_250_hexbins(\d+)\.csv$",           re.IGNORECASE),
     "spei_"),
    ("ppt",    "grids",
     data_external / "Precip" / "250",
     re.compile(r"^Precip_250_grids(\d+)\.csv$",           re.IGNORECASE),
     "ppt_"),
    ("ppt",    "hexbins",
     data_external / "Precip" / "250",
     re.compile(r"^CHIRPS_250_hexbins(\d+)\.csv$",         re.IGNORECASE),
     "ppt_"),
    ("lst",    "grids",
     data_external / "LST" / "250",
     re.compile(r"^LST_250_grids(\d+)\.csv$",              re.IGNORECASE),
     "lst_"),
    ("lst",    "hexbins",
     data_external / "LST" / "250",
     re.compile(r"^LST_250_hexbins(\d+)\.csv$",            re.IGNORECASE),
     "lst_"),
    ("hansen", "grids",
     data_external / "Hansen_Forest" / "250",
     re.compile(r"^HANSEN_250_grids(\d+)\.csv$",           re.IGNORECASE),
     "hansen_"),
    ("hansen", "hexbins",
     data_external / "Hansen_Forest" / "250",
     re.compile(r"^HANSEN_250_hexbins(\d+)\.csv$",         re.IGNORECASE),
     "hansen_"),
    ("wc_ag",  "grids",
     data_external / "Crop Cover" / "250",
     re.compile(r"^WC_AGRICULTURE_250_grids(\d+)\.csv$",   re.IGNORECASE),
     "wc_ag_"),
    ("wc_ag",  "hexbins",
     data_external / "Crop Cover" / "250",
     re.compile(r"^WC_AGRICULTURE_250_hexbins(\d+)\.csv$", re.IGNORECASE),
     "wc_ag_"),
]

# Flat (non-chunked) AG+DEM source files.
# Using "csv_" prefix to match the existing CANONICAL_GRID_BIG / CANONICAL_HEX_BIG maps.
AG_DEM_250 = {
    "grids":   data_external / "AG_DEM_250_grids.csv",
    "hexbins": data_external / "AG_DEM_250_hexbins.csv",
}

# geometry table for each geom_type at 250k resolution
TARGET_TABLES_250 = {
    "grids":   "grid_250k_v3",
    "hexbins": "hex_250k",
}


# ----------------------------
# multi-source join helper
# ----------------------------

def create_joined_output_multi_sources(
    geom_table: str,
    sources: list,          # [(schema, table, col_prefix), ...]
    output_schema: str,
    output_table: str,
):
    """
    Create a VIEW joining the geometry table to any number of accumulation tables.

    Using a view rather than materializing a very wide table avoids PostgreSQL's
    row-size limit while preserving every geometry-table attribute and every joined
    source column for the downstream export notebook.
    """
    ensure_schema(output_schema)
    drop_relation(output_schema, output_table)

    # Keep all geometry table columns in the view.
    select_parts = [sql.SQL("v.*")]
    join_parts   = []

    total_cols = len(get_columns(SCHEMA, geom_table))

    for i, (src_schema, src_table, col_prefix) in enumerate(sources):
        alias_str = f"s{i}"
        src_cols  = [c for c in get_columns(src_schema, src_table) if c != ID_COL]
        total_cols += len(src_cols)

        if src_cols:
            # Cast each text column to double precision for cleaner downstream typing.
            # NULLIF handles empty strings that came from the CSV staging step.
            col_exprs = sql.SQL(", ").join(
                sql.SQL("NULLIF({}.{}, '')::double precision AS {}").format(
                    sql.Identifier(alias_str),
                    sql.Identifier(c),
                    sql.Identifier(f"{col_prefix}{c}"),
                )
                for c in src_cols
            )
            select_parts.append(col_exprs)

        join_parts.append(
            sql.SQL(
                "\n    LEFT JOIN {}.{} AS {}"
                "\n      ON v.{} = NULLIF({}.{}, '')::integer"
            ).format(
                sql.Identifier(src_schema),
                sql.Identifier(src_table),
                sql.Identifier(alias_str),
                sql.Identifier(ID_COL),
                sql.Identifier(alias_str),
                sql.Identifier(ID_COL),
            )
        )

    print(f"  creating view with ~{total_cols} columns")

    create_view = sql.SQL(
        "CREATE VIEW {}.{} AS\n"
        "SELECT {}\n"
        "FROM {}.{} AS v{};"
    ).format(
        sql.Identifier(output_schema),
        sql.Identifier(output_table),
        sql.SQL(", ").join(select_parts),
        sql.Identifier(SCHEMA),
        sql.Identifier(geom_table),
        sql.SQL("").join(join_parts),
    )

    run_sql(create_view)


# ----------------------------
# 250k pipeline
# ----------------------------

def process_250k_sources():
    ensure_schema(STAGING_SCHEMA)

    for geom_type in ("grids", "hexbins"):
        geom_table = TARGET_TABLES_250[geom_type]
        out_table  = f"{OUTPUT_PREFIX}{geom_table}"   # joined_grid_250k_v3 / joined_hex_250k

        print(f"\n{'='*60}")
        print(f"250k {geom_type}  ->  {OUTPUT_SCHEMA}.{out_table}")
        print('='*60)

        try:
            all_sources = []   # (schema, accum_table, col_prefix)

            # ---- 1) flat AG+DEM file ("csv_" prefix for canonical-map compatibility) ----
            ag_dem_path = AG_DEM_250[geom_type]
            ag_dem_acc  = f"acc_ag_dem_250_{geom_type}"
            print(f"\n  [ag_dem] {ag_dem_path.name}  ->  {STAGING_SCHEMA}.{ag_dem_acc}")
            load_flat_to_accum(ag_dem_path, STAGING_SCHEMA, ag_dem_acc)
            all_sources.append((STAGING_SCHEMA, ag_dem_acc, "csv_"))

            # ---- 2) chunked sources ----
            for (src_name, src_geom_type, src_dir, src_re, col_prefix) in CHUNK_SOURCE_CONFIGS_250:
                if src_geom_type != geom_type:
                    continue

                acc_table = f"acc_{src_name}_250_{geom_type}"
                chunks    = sorted_chunks(src_dir, src_re)

                if not chunks:
                    print(f"  [WARN] no chunks found for {src_name}/{geom_type} in {src_dir}")
                    continue

                print(f"\n  [{src_name}] {len(chunks)} chunks  ->  {STAGING_SCHEMA}.{acc_table}")

                # infer filtered column list from first chunk header
                first_cols = get_data_cols(chunks[0])
                ensure_accum_table(STAGING_SCHEMA, acc_table, first_cols)

                for i, csv_path in enumerate(chunks, 1):
                    stg = f"stg_{src_name}_250_{geom_type}_{i}"
                    print(f"    chunk {i}/{len(chunks)}: {csv_path.name}")
                    kept = copy_csv_to_table_filtered(csv_path, STAGING_SCHEMA, stg)
                    upsert_chunk_into_accum(kept, STAGING_SCHEMA, stg, STAGING_SCHEMA, acc_table)
                    drop_table(STAGING_SCHEMA, stg)

                all_sources.append((STAGING_SCHEMA, acc_table, col_prefix))

            # ---- 3) final multi-source join ----
            print(f"\n  creating view {OUTPUT_SCHEMA}.{out_table} ...")
            create_joined_output_multi_sources(
                geom_table, all_sources, OUTPUT_SCHEMA, out_table
            )

            conn.commit()
            print(f"  done view: {OUTPUT_SCHEMA}.{out_table}")

        except Exception as e:
            conn.rollback()
            print(f"FAILED 250k {geom_type}: {e}")
            raise

In [17]:
# ----------------------------
# main routines  (15k pipeline — unchanged)
# ----------------------------
def group_dem_chunks(dem_dir: Path):
    """
    Returns dict[(res:int, type:str)] -> list[Path] sorted by chunk number
    """
    groups = defaultdict(list)
    for p in dem_dir.glob("*.csv"):
        m = DEM_RE.match(p.name)
        if not m:
            continue
        res = int(m.group(1))
        typ = m.group(2).lower()
        chunk = int(m.group(3))
        groups[(res, typ)].append((chunk, p))

    out = {}
    for k, lst in groups.items():
        out[k] = [p for _, p in sorted(lst, key=lambda x: x[0])]
    return out

def process_dem_chunks():
    ensure_schema(STAGING_SCHEMA)

    dem_groups = group_dem_chunks(DEM_DIR)
    if not dem_groups:
        raise FileNotFoundError(f"No DEM chunks found in {DEM_DIR}")

    for (res, typ), files in dem_groups.items():
        if (res, typ) not in TARGET_TABLES:
            raise KeyError(f"No target table mapping for DEM ({res}, {typ}). Add it to TARGET_TABLES.")

        target_table = TARGET_TABLES[(res, typ)]
        acc_table = f"acc_dem_{res}_{typ}"

        print(f"\n=== DEM {res} {typ}: {len(files)} chunks -> accumulate {STAGING_SCHEMA}.{acc_table} -> join with {SCHEMA}.{target_table}")

        try:
            first_header = files[0].read_text(encoding="utf-8", errors="replace").splitlines()[0]
            cols = [c.strip() for c in first_header.split(",")]
            if ID_COL not in cols:
                raise ValueError(f"{files[0].name} missing '{ID_COL}'")

            ensure_accum_table(STAGING_SCHEMA, acc_table, cols)

            for i, csv_path in enumerate(files, start=1):
                stg_table = f"stg_{csv_path.stem.lower()}"
                print(f"  - chunk {i}/{len(files)}: {csv_path.name} -> {STAGING_SCHEMA}.{stg_table} -> upsert to {STAGING_SCHEMA}.{acc_table}")
                copy_csv_to_table(csv_path, STAGING_SCHEMA, stg_table)
                upsert_chunk_into_accum(cols, STAGING_SCHEMA, stg_table, STAGING_SCHEMA, acc_table)
                drop_table(STAGING_SCHEMA, stg_table)

            ag_csv = data_external / f"AG_{res}_{typ}.csv"
            ag_acc_table = f"acc_ag_{res}_{typ}"

            if not ag_csv.exists():
                raise FileNotFoundError(f"Missing AG CSV for ({res},{typ}): {ag_csv}")

            load_ag_to_accum(ag_csv, STAGING_SCHEMA, ag_acc_table)

            out_table = f"{OUTPUT_PREFIX}{target_table}"
            create_joined_output_two_sources(
                view_name=target_table,
                dem_schema=STAGING_SCHEMA,
                dem_table=acc_table,
                ag_schema=STAGING_SCHEMA,
                ag_table=ag_acc_table,
                output_schema=OUTPUT_SCHEMA,
                output_table=out_table,
                dem_prefix="dem_",
                ag_prefix="ag_",
            )

            conn.commit()
            print(f"Done combined join: {OUTPUT_SCHEMA}.{out_table}")

        except Exception as e:
            conn.rollback()
            print(f"FAILED DEM {res} {typ}: {e}")
            raise

def parse_ag_filename(name: str):
    m = re.match(r"^AG_(\d+)_((?:grids)|(?:hexbins))\.csv$", name, re.IGNORECASE)
    if not m:
        return None
    return int(m.group(1)), m.group(2).lower()


# ----------------------------
# entry points
# ----------------------------
# Run 15k DEM+AG pipeline:
#   process_dem_chunks()
#
# Run 250k multi-source pipeline:
process_250k_sources()

conn.close()  # close when all pipelines are done


250k grids  ->  public.joined_grid_250k_v3

  [ag_dem] AG_DEM_250_grids.csv  ->  scratch.acc_ag_dem_250_grids

  [spei] 15 chunks  ->  scratch.acc_spei_250_grids
    chunk 1/15: SPEI_250_grids0.csv
    chunk 2/15: SPEI_250_grids1.csv
    chunk 3/15: SPEI_250_grids2.csv
    chunk 4/15: SPEI_250_grids3.csv
    chunk 5/15: SPEI_250_grids4.csv
    chunk 6/15: SPEI_250_grids5.csv
    chunk 7/15: SPEI_250_grids6.csv
    chunk 8/15: SPEI_250_grids7.csv
    chunk 9/15: SPEI_250_grids8.csv
    chunk 10/15: SPEI_250_grids9.csv
    chunk 11/15: SPEI_250_grids10.csv
    chunk 12/15: SPEI_250_grids11.csv
    chunk 13/15: SPEI_250_grids12.csv
    chunk 14/15: SPEI_250_grids13.csv
    chunk 15/15: SPEI_250_grids14.csv

  [ppt] 15 chunks  ->  scratch.acc_ppt_250_grids
    chunk 1/15: Precip_250_grids0.csv
    chunk 2/15: Precip_250_grids1.csv
    chunk 3/15: Precip_250_grids2.csv
    chunk 4/15: Precip_250_grids3.csv
    chunk 5/15: Precip_250_grids4.csv
    chunk 6/15: Precip_250_grids5.csv
    chun